# Relation Pipeline
Notebook này huấn luyện riêng relation classifier với tên file và biến rõ ràng theo relation pipeline.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import torch
from torch.utils.data import DataLoader

current_dir = os.path.abspath(os.getcwd())
project_root = os.path.dirname(current_dir) if current_dir.endswith('notebooks') else current_dir
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from src.utils.config_loader import load_task_configs, print_config
from src.data.download import download_and_extract_metadata, download_vg_images, verify_download
from src.data.preprocessing import build_relation_vocab_and_splits
from src.features.relation_feature_extractor import extract_relation_features
from src.data.relation_dataset import build_relation_datasets
from src.models.relation import RelationClassifier
from src.training.relation_trainer import RelationTrainer
from src.evaluation.metrics import compute_classification_metrics
from src.utils.memory import cleanup_cuda_memory

base_config = load_task_configs('configs/config.yaml')
relation_config = load_task_configs('configs/config.yaml', 'configs/relation_config.yaml')

project_root_path = Path.cwd()
raw_dir = project_root_path / base_config.paths.raw_dir
image_dir = project_root_path / base_config.dataset.image_dir
processed_dir = project_root_path / relation_config.dataset.processed_dir
feature_dir = processed_dir / relation_config.dataset.feature_cache_dir

device = 'cuda' if str(base_config.device).lower().startswith('cuda') and torch.cuda.is_available() else 'cpu'

def require_files(paths, label):
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f'Thiếu {label}: {missing}')


def ensure_nonempty_cache(cache_path):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return False
    if cache_path.stat().st_size == 0:
        return False
    try:
        cache = torch.load(cache_path, map_location='cpu')
        return bool(cache)
    except Exception:
        return False


def cache_output(split_name, input_mode):
    suffix = None if input_mode == 'rgb' else input_mode
    filename = f'{split_name}_features.pt' if suffix is None else f'{split_name}_{suffix}_features.pt'
    return feature_dir / filename


def collect_split_image_ids(processed_root, split_names):
    image_ids = set()
    for split_name in split_names:
        annotation_file = Path(processed_root) / split_name / 'annotations.json'
        if not annotation_file.exists():
            raise FileNotFoundError(f'Thiếu annotation: {annotation_file}')

        with open(annotation_file, 'r', encoding='utf-8') as f:
            raw = json.load(f)

        samples = raw if isinstance(raw, list) else raw.get('samples', [])
        image_ids.update(sample['image_id'] for sample in samples)

    return sorted(image_ids)


def ensure_images_for_splits(pipeline_name, processed_root, split_names):
    image_ids = collect_split_image_ids(processed_root, split_names)
    missing_ids = [img_id for img_id in image_ids if not (image_dir / f'{img_id}.jpg').exists()]

    print(f'[{pipeline_name}] Tổng ảnh tham chiếu: {len(image_ids)} | thiếu local: {len(missing_ids)}')
    if missing_ids:
        print(f'[{pipeline_name}] Đang tải bổ sung ảnh còn thiếu...')
        downloaded = download_vg_images(missing_ids, image_dir=str(image_dir))
        if len(downloaded) < len(missing_ids):
            print(f'[Warning] {pipeline_name}: còn {len(missing_ids) - len(downloaded)} ảnh chưa tải được.')


def normalize_input_mode(mode):
    mode = str(mode).lower().strip()
    aliases = {
        'grayscale': 'gray',
        'grey': 'gray',
        'edge': 'contour',
        'edges': 'contour',
    }
    mode = aliases.get(mode, mode)
    if mode not in {'rgb', 'gray', 'contour'}:
        raise ValueError("input mode phải là 'rgb', 'gray', hoặc 'contour'")
    return mode


def evaluate_relation_model(model, loader):
    model.eval()
    logits_list = []
    targets_list = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            features = batch.get('feature')
            if features is None:
                features = batch.get('image')
            if features is None:
                features = batch.get('union_image')
            logits = model(features, batch['spatial'])
            logits_list.append(logits.cpu())
            targets_list.append(batch['relation_label'].cpu())

    return compute_classification_metrics(torch.cat(logits_list), torch.cat(targets_list), num_classes=relation_train_ds.num_relations)


download_data = bool(base_config.pipeline.download_data)
strict_sample_mode = bool(base_config.sampling.strict_mode)
sample_size = int(base_config.sampling.sample_size)
sample_seed = int(base_config.sampling.seed)
image_download_mode = 'none' if strict_sample_mode else str(base_config.sampling.image_download_mode)
pre_extract_features = bool(base_config.pipeline.pre_extract_features)
feature_batch_size = int(base_config.feature_extraction.batch_size)
feature_resize_size = int(base_config.feature_extraction.resize_size)
feature_crop_size = int(base_config.feature_extraction.crop_size)
feature_mean = [float(x) for x in base_config.image.mean]
feature_std = [float(x) for x in base_config.image.std]
full_split_ratios = (float(base_config.split.train), float(base_config.split.val), float(base_config.split.test))
preprocess_split_ratios = tuple(float(x) for x in base_config.sampling.split_ratios) if strict_sample_mode else full_split_ratios
max_samples = None if base_config.sampling.max_samples is None else int(base_config.sampling.max_samples)

relation_learnable_backbone = bool(relation_config.backbone.learnable_backbone)
relation_use_cache = bool(relation_config.dataset.use_feature_cache) and not relation_learnable_backbone
relation_input_mode = normalize_input_mode(relation_config.dataset.input_mode)
relation_batch_size = int(relation_config.training.batch_size)
relation_lr = float(relation_config.training.lr)
relation_max_epochs = int(relation_config.training.max_epochs)
relation_roi_size = int(base_config.image.roi_size)
relation_hidden_dim = int(relation_config.model.hidden_dim)
relation_dropout = float(relation_config.model.dropout)
relation_num_layers = int(relation_config.model.num_layers)
relation_attention_heads = int(relation_config.model.attention_heads)
relation_fusion_method = str(relation_config.model.fusion_method)

print_config(base_config)
print_config(relation_config)
print(f'Project root: {project_root_path}')
print(f'Using device: {device}')
print(f'Pipeline: Relation')
print(f'Strict sample mode: {strict_sample_mode}')
print(f'Sample size: {sample_size} | split ratios: {preprocess_split_ratios} | seed: {sample_seed}')
print(f'Processed root: {processed_dir}')
print(f'Feature extraction batch size: {feature_batch_size}')
print(f'Relation learnable backbone: {relation_learnable_backbone}')
print(f'Relation input mode: {relation_input_mode}')
print(f'Relation batch size: {relation_batch_size}, lr: {relation_lr}, max_epochs: {relation_max_epochs}')

CONFIGURATION
project_name: ImageCaptioningE2E
seed: 42
device: cuda
paths:
  data_dir: data
  raw_dir: data/raw
  processed_dir: data/processed
  feature_dir: data/features
  checkpoint_dir: checkpoints
  log_dir: logs
dataset:
  repo_id: ranjaykrishna/visual_genome
  cache_dir: data/raw/hf_cache
  image_dir: data/raw/images
image:
  roi_size: 224
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
split:
  train: 0.7
  val: 0.15
  test: 0.15
training:
  batch_size: 1024
  num_workers: 2
  pin_memory: true
  max_epochs: 30
  early_stopping_patience: 5
  gradient_clip_val: 1.0
  log_every_n_steps: 0
optimizer:
  name: adamw
  lr: 0.0001
  weight_decay: 0.0001
scheduler:
  name: cosine
  warmup_epochs: 2
logging:
  use_tensorboard: true
  use_wandb: false
  wandb_project: vg-caption
pipeline:
  training_mode: all
  download_data: true
  pre_extract_features: true
sampling:
  strict_mode: true
  sample_size: 200
  split_ratios:
  - 0.8
  - 0.1
  - 0.1
  seed: 42
  

In [ ]:
# Load or verify raw data
if download_data:
    print('Downloading metadata for the relation pipeline...')
    download_and_extract_metadata(raw_dir=str(raw_dir), keep_zip=False)

    raw_status = verify_download(raw_dir=str(raw_dir))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(f'Thiếu metadata sau khi tải: {missing_metadata}')

    with open(raw_dir / 'image_data.json', 'r', encoding='utf-8') as f:
        image_data = json.load(f)

    if image_download_mode == 'sample':
        sample_ids = [img['image_id'] for img in image_data[:sample_size]]
        print(f'Bắt đầu tải bộ sample {len(sample_ids)} ảnh...')
        downloaded_images = download_vg_images(sample_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError('Không tải được ảnh mẫu nào.')
    elif image_download_mode == 'all':
        all_ids = [img['image_id'] for img in image_data]
        print(f'Bắt đầu tải toàn bộ {len(all_ids)} ảnh...')
        downloaded_images = download_vg_images(all_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError('Không tải được ảnh nào.')
    elif image_download_mode == 'none':
        print('Bỏ qua tải ảnh theo cấu hình.')
    else:
        raise ValueError("IMAGE_DOWNLOAD_MODE phải là 'none', 'sample', hoặc 'all'.")
else:
    print('Bỏ qua tải mới, chỉ kiểm tra dữ liệu hiện có...')
    raw_status = verify_download(raw_dir=str(raw_dir))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(
            'Thiếu dữ liệu RAW cần thiết: ' + ', '.join(missing_metadata) +
            '. Hãy bật DOWNLOAD_DATA = True hoặc đặt đúng thư mục data/raw.'
        )

if strict_sample_mode:
    image_data_file = raw_dir / 'image_data.json'
    require_files([image_data_file], 'image_data.json')

    with open(image_data_file, 'r', encoding='utf-8') as f:
        image_data = json.load(f)

    all_image_ids = [img['image_id'] for img in image_data]
    sample_count = min(sample_size, len(all_image_ids))
    sample_image_ids = random.Random(sample_seed).sample(all_image_ids, sample_count)
    print(f'Đã chọn trước {len(sample_image_ids)} image_id cho sample strict.')
else:
    sample_image_ids = None
    print('Bỏ qua strict sample; sẽ dùng toàn bộ dữ liệu theo split mặc định.')

build_relation_vocab_and_splits(
    raw_dir=str(raw_dir),
    processed_dir=str(processed_dir),
    max_relations=int(relation_config.labels.max_relations),
    sample_image_ids=sample_image_ids if strict_sample_mode else None,
    split_by_image_id=strict_sample_mode,
    split_ratios=preprocess_split_ratios,
    seed=sample_seed,
)

require_files(
    [
        processed_dir / 'relation_vocab.json',
        processed_dir / 'train' / 'annotations.json',
        processed_dir / 'val' / 'annotations.json',
        processed_dir / 'test' / 'annotations.json',
    ],
    'Relation pipeline processed files',
)

if pre_extract_features and relation_use_cache:
    feature_dir.mkdir(parents=True, exist_ok=True)
    ensure_images_for_splits('Relation', processed_dir, ['train', 'val', 'test'])

    print('Extracting relation features...')
    for split_name in ['train', 'val', 'test']:
        relation_output = cache_output(split_name, relation_input_mode)
        extract_relation_features(
            annotation_file=str(processed_dir / split_name / 'annotations.json'),
            image_dir=str(image_dir),
            output_file=str(relation_output),
            backbone=str(relation_config.backbone.name),
            pretrained=bool(relation_config.backbone.pretrained),
            batch_size=feature_batch_size,
            device=device,
            resize_size=feature_resize_size,
            crop_size=feature_crop_size,
            mean=feature_mean,
            std=feature_std,
            input_mode=relation_input_mode,
        )
        if not ensure_nonempty_cache(relation_output):
            raise RuntimeError(f'Relation feature cache rỗng: {relation_output}')
else:
    print('Bỏ qua feature extraction theo cấu hình PRE_EXTRACT_FEATURES hoặc learnable backbone.')

relation_train_ds, relation_val_ds, relation_test_ds = build_relation_datasets(
    processed_dir=str(processed_dir),
    image_dir=str(image_dir),
    roi_size=relation_roi_size,
    use_feature_cache=relation_use_cache,
    use_spatial_features=bool(relation_config.spatial.use_spatial_features),
    feature_cache_dir=str(feature_dir),
    input_mode=relation_input_mode,
    max_samples=max_samples,
    train_horizontal_flip_p=float(relation_config.augmentation.random_horizontal_flip),
    train_color_jitter=bool(relation_config.augmentation.color_jitter.enabled),
    train_brightness=float(relation_config.augmentation.color_jitter.brightness),
    train_contrast=float(relation_config.augmentation.color_jitter.contrast),
    train_saturation=float(relation_config.augmentation.color_jitter.saturation),
    train_hue=float(relation_config.augmentation.color_jitter.hue),
    train_random_erasing_p=float(relation_config.augmentation.random_erasing_p),
    train_resize_delta=int(relation_config.augmentation.resize_delta),
    mean=feature_mean,
    std=feature_std,
)

relation_model = RelationClassifier(
    num_relations=relation_train_ds.num_relations,
    feature_dim=int(relation_config.backbone.feature_dim),
    spatial_dim=int(relation_config.spatial.spatial_dim),
    hidden_dim=relation_hidden_dim,
    dropout=relation_dropout,
    num_layers=relation_num_layers,
    attention_heads=relation_attention_heads,
    fusion_method=relation_fusion_method,
    backbone_name=str(relation_config.backbone.name) if relation_learnable_backbone else None,
    pretrained=bool(relation_config.backbone.pretrained),
    freeze_backbone=bool(relation_config.backbone.freeze_backbone),
    learnable_backbone=relation_learnable_backbone,
    device=device,
)

relation_optimizer = torch.optim.AdamW(
    relation_model.parameters(),
    lr=relation_lr,
    weight_decay=float(base_config.optimizer.weight_decay),
)

relation_train_loader = DataLoader(
    relation_train_ds,
    batch_size=relation_batch_size,
    shuffle=True,
    num_workers=int(base_config.training.num_workers),
    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),
)
relation_val_loader = DataLoader(
    relation_val_ds,
    batch_size=relation_batch_size,
    shuffle=False,
    num_workers=int(base_config.training.num_workers),
    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),
)
relation_test_loader = DataLoader(
    relation_test_ds,
    batch_size=relation_batch_size,
    shuffle=False,
    num_workers=int(base_config.training.num_workers),
    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),
)

relation_trainer = RelationTrainer(
    model=relation_model,
    train_loader=relation_train_loader,
    val_loader=relation_val_loader,
    optimizer=relation_optimizer,
    label_smoothing=float(relation_config.loss.label_smoothing),
    use_auto_class_weights=True,
    freeze_backbone=bool(relation_config.backbone.freeze_backbone),
    freeze_epochs=int(relation_config.backbone.freeze_epochs),
    max_epochs=relation_max_epochs,
    early_stopping_patience=int(base_config.training.early_stopping_patience),
    gradient_clip_val=float(base_config.training.gradient_clip_val),
    log_every_n_steps=int(base_config.training.log_every_n_steps),
    device=device,
    use_amp=(device == 'cuda'),
)

print('Bắt đầu training relation pipeline...')
relation_val_metrics = relation_trainer.train()
relation_best_checkpoint = relation_trainer.checkpoint_manager.get_best_checkpoint_path()
print('Relation pipeline completed.')
print(f'Best relation checkpoint: {relation_best_checkpoint}')

relation_model = relation_trainer.model
relation_test_metrics = evaluate_relation_model(relation_model, relation_test_loader)
print('Relation test metrics:')
print(relation_test_metrics)

In [ ]:
cleanup_cuda_memory(note='Relation pipeline notebook finished')